# Chapter 14. GPU 가속 시뮬레이션 — 1,000개 진자를 하나의 배열로

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter14_2_gpu_batch_sim.ipynb)

책 본문: [14.2절](https://smhanlab.com/book-ml/kor/ml2/chapter14.html)

이 노트북은 Isaac Sim의 핵심 데이터 구조 — "수천 개 환경을 동시에 돌린다" — 를
numpy 하나로 재현합니다. 14.1의 진자 N개의 상태를 **하나의 배열** `theta[N]`에 담아,
하나의 벡터화 numpy 연산으로 전체를 *동시에* 업데이트하는 것입니다.
그 구조의 효과를 "CPU 순차(진자를 하나씩 Python 루프)"와 "배치(numpy 벡터화, 전수 동시)"
의 벽 시간 실측으로 직접 비교합니다.

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import numpy as np
import time
import math

IMG = "/home/smhan/book-ml/kor/src/images"

## 1. `step_batch`: N개 진자의 물리를 numpy 연산 한 번으로 (RK4)

14.1의 진자 $\ddot\theta = -(g/\ell)\sin\theta - b\dot\theta + \tau/I$ 의 물리 스텝을
RK4로 $\Delta t = 0.002$ s 전진시키는 함수입니다. `theta`가 (N,) 배열이면,
`f` 안의 `np.sin`, `b*thd` 등 *모든* 연산이 N개 요소에 동시에 적용됩니다 —
이것이 "커널 한 번"의 CPU 위 미니판입니다 (GPU에서는 같은 배열 연산을
수천 개 코어가 한 번에 수행합니다).

In [2]:
def step_batch(theta, thetadot, torque, dt=0.002, b=0.1, L=0.5, m=1.0):
    """N개 진자의 물리를 한 번의 numpy 연산으로 동시 스텝 (RK4)."""
    def f(th, thd, tau):
        return thd, -(9.81/L)*np.sin(th) - b*thd + tau  # 각가속도
    # RK4: 4단계 미분 계산 — 각 단계가 numpy 배열 전체에 동시에 적용
    k1t, k1v = f(theta, thetadot, torque)
    k2t, k2v = f(theta+0.5*dt*k1t, thetadot+0.5*dt*k1v, torque)
    k3t, k3v = f(theta+0.5*dt*k2t, thetadot+0.5*dt*k2v, torque)
    k4t, k4v = f(theta+dt*k3t, thetadot+dt*k3v, torque)
    theta   += dt/6*(k1t + 2*k2t + 2*k3t + k4t)
    thetadot+= dt/6*(k1v + 2*k2v + 2*k3v + k4v)
    return theta, thetadot

## 2. 실측: CPU 순차 vs 배치

N=1,000개 진자, 2,000스텝(= 4s 시뮬레이션), 같은 초깃값, 같은 PD 토크
($\tau = -k_p\theta - k_d\dot\theta$) 조건에서, 두 실행 방식의 벽 시간을 재봅니다.

- **CPU 순차**: 진자를 *하나씩* Python 루프로 스텝 — 진자마다 함수 호출,
  인덱싱 오버헤드가 매 스텝마다 붙습니다.
- **배치**: 1,000개 진자를 하나의 배열에 담아 numpy 벡터화 연산으로
  전체를 동시에 업데이트 — 스텝마다 오버헤드 *한 번*만 치릅니다.

총 연산량(N × T 개의 환경 스텝)은 동일합니다. 차이는 오직 *구조*뿐입니다.

In [3]:
N, T = 1000, 2000   # 1,000개 진자, 2,000스텝 = 4s 시뮬레이션
dt = 0.002
kp, kd = 4.0, 1.5  # PD 이득 (θ=0으로 감쇠 수렴)

rng = np.random.default_rng(0)
theta0 = rng.uniform(-2.5, 2.5, size=N)

def step_scalar(th, thd, tau, dt=0.002, b=0.1, L=0.5):
    """진자 하나(스칼라)의 RK4 한 스텝 — 'CPU 순차' 쪽."""
    def f(t, v, u):
        return v, -(9.81/L)*math.sin(t) - b*v + u
    k1t, k1v = f(th, thd, tau)
    k2t, k2v = f(th+0.5*dt*k1t, thd+0.5*dt*k1v, tau)
    k3t, k3v = f(th+0.5*dt*k2t, thd+0.5*dt*k2v, tau)
    k4t, k4v = f(th+dt*k3t, thd+dt*k3v, tau)
    return th + dt/6*(k1t + 2*k2t + 2*k3t + k4t), thd + dt/6*(k1v + 2*k2v + 2*k3v + k4v)

# (a) CPU 순차: 진자 하나씩, 각각 2,000스텝 (배치와 동일한 총 연산량)
t_start = time.perf_counter()
for i in range(N):
    th, thd = float(theta0[i]), 0.0
    for _ in range(T):
        tau = -kp*th - kd*thd
        th, thd = step_scalar(th, thd, tau)
t_seq = time.perf_counter() - t_start

# (b) 배치: 1,000개 진자를 하나의 배열로, 한 번의 numpy 연산으로 전수 동시
theta, thetadot = theta0.copy(), np.zeros(N)
t_start = time.perf_counter()
for _ in range(T):
    tau = -kp*theta - kd*thetadot
    theta, thetadot = step_batch(theta, thetadot, tau, dt=0.002)
t_batch = time.perf_counter() - t_start

print(f"CPU 순차: {t_seq:6.2f} s   {N*T/t_seq:,.0f} 환경스텝/초")
print(f"배치(numpy): {t_batch:6.2f} s   {N*T/t_batch:,.0f} 환경스텝/초")
print(f"가속 배수(순차/배치): {t_seq/t_batch:.1f}배")

CPU 순차:   1.51 s   1,321,211 환경스텝/초
배치(numpy):   0.10 s   20,642,876 환경스텝/초
가속 배수(순차/배치): 15.6배


## 3. 팬아웃: 같은 코드, 다른 출발점

1,000개 진자를 $\theta_0 \in [-2.5, 2.5]$ 균등분포로 무작위 시작해 2,000스텝 굴립니다.
모든 궤적은 *같은* 물리 법칙(같은 코드)을 따르지만, *출발점(상태)*이 다르므로
궤적의 위상·진폭이 다릅니다 — "공유하는 것은 코드, 달라지는 것은 상태"의
직접 증거입니다. 하단 히스토그램은 4초 뒤 분포가 0 중심으로 수렴하는 것을 보여줍니다.
(CNN이 같은 필터를 모든 위치에 공유하는 것과 같은 구조, ML1 10.1.)

In [4]:
traj = np.empty((T+1, N))
traj[0] = theta0
th, thd = theta0.copy(), np.zeros(N)
for t in range(T):
    tau = -kp*th - kd*thd
    th, thd = step_batch(th, thd, tau, dt=0.002)
    traj[t+1] = th

t_arr = np.arange(T+1) * dt
fig, axes = plt.subplots(2, 1, figsize=(9, 7))

ax = axes[0]
for i in range(20):                      # 회색: 일부 궤적 20개
    ax.plot(t_arr, traj[:, i], color="gray", alpha=0.35, lw=0.8)
for i in range(5):                       # 파랑: 처음 5개
    ax.plot(t_arr, traj[:, i], color="tab:blue", lw=1.6)
ax.set_xlabel("시간 (s)")
ax.set_ylabel(r"$\theta$ (rad)")
ax.set_title("1,000개 진자: 같은 물리 법칙(코드), 다른 초깃값(상태)")
ax.grid(alpha=0.3)

ax = axes[1]
ax.hist(theta0, bins=50, density=True, alpha=0.6, color="tab:blue", label="초기 $\theta_0$ (균등분포)")
ax.hist(th, bins=50, density=True, alpha=0.6, color="tab:red", label="4s 후 $\theta$")
ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_xlabel(r"$\theta$ (rad)")
ax.set_ylabel("밀도")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(IMG + "/ch14_2_batch_fan.svg", bbox_inches="tight")
print("저장됨:", IMG + "/ch14_2_batch_fan.svg")

저장됨: /home/smhan/book-ml/kor/src/images/ch14_2_batch_fan.svg


## 4. 확인 — 무엇이 달라졌는가

- 가속 배수의 정체: 물리 계산이 빨라진 것이 아니라, **(a)** 진자마다 붙던
  Python 함수 호출/인덱싱 오버헤드가 사라지고, **(b)** 하나의 배열 연산 안의
  연산이 N개 진자만큼 나눠진 것입니다. 물리 계산 자체는 같은 CPU에서 그대로 합니다.
- **배치는 데이터 구조의 필요조건**입니다 — 없으면 GPU 병렬 자체를 할 수 없지만,
  numpy가 CPU의 8~16 SIMD 레지스터로 처리하는 *같은* 배열 연산을 GPU는
  *수천* 코어가 한 번에 수행하는 것이 진짜 GPU 병렬(Isaac Sim)입니다.
- 환경 스텝을 모으는 속도가 N배가 되었다고 *정책 업데이트*가 N배 빨라지는 것은
  아닙니다 (수집 속도와 사용 효율은 별개 축 — 본문 "자주 하는 실수" 참고).